# DD2424 Character-Level Text Generation

This notebook is the whole project in one place.
It keeps the code simple and readable while covering the required pipeline:

- Shakespeare preprocessing
- Vanilla RNN, 1-layer LSTM, and 2-layer LSTM
- Training, validation, and testing
- Text generation with temperature and top-p sampling
- Perplexity, BLEU, and plots
- Small experiment sweeps saved to CSV


## 1. Setup

We keep the setup small: standard PyTorch, NumPy, matplotlib, NLTK, and a few helpers from the Python standard library.


In [8]:
from __future__ import annotations

import csv
import math
import random
from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Iterable

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nltk
import numpy as np
import torch
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Paths
ROOT = Path(".").resolve()
TEXT_PATH = ROOT / "shakespeare.txt"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

## 2. Data preprocessing

This section loads Shakespeare, builds a character vocabulary, splits the text into train/validation/test sets, and creates simple sequence batches.


In [9]:
@dataclass
class ShakespeareCorpus:
    text: str
    stoi: dict[str, int]
    itos: dict[int, str]

    @property
    def vocab_size(self) -> int:
        return len(self.stoi)

    @classmethod
    def from_file(cls, path: Path) -> "ShakespeareCorpus":
        text = Path(path).read_text(encoding="utf-8")
        vocab = sorted(set(text))
        stoi = {ch: i for i, ch in enumerate(vocab)}
        itos = {i: ch for ch, i in stoi.items()}
        return cls(text=text, stoi=stoi, itos=itos)

    def encode(self, text: str) -> list[int]:
        return [self.stoi[ch] for ch in text]

    def decode(self, indices: Iterable[int] | torch.Tensor) -> str:
        if isinstance(indices, torch.Tensor):
            indices = indices.detach().cpu().tolist()
        return "".join(self.itos[int(i)] for i in indices)

    def split(self, train_ratio: float = 0.8, val_ratio: float = 0.1) -> dict[str, str]:
        total = len(self.text)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)
        return {
            "train": self.text[:train_end],
            "validation": self.text[train_end:val_end],
            "test": self.text[val_end:],
        }


class CharSequenceDataset(Dataset):
    def __init__(self, encoded_text: list[int], seq_len: int, stride: int | None = None) -> None:
        self.encoded_text = encoded_text
        self.seq_len = seq_len
        self.stride = stride or seq_len
        self.max_start = len(encoded_text) - seq_len - 1
        if self.max_start <= 0:
            raise ValueError("Text is too short for this sequence length.")

    def __len__(self) -> int:
        return max(1, self.max_start // self.stride + 1)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = index * self.stride
        if start > self.max_start:
            start = self.max_start
        end = start + self.seq_len
        x = torch.tensor(self.encoded_text[start:end], dtype=torch.long)
        y = torch.tensor(self.encoded_text[start + 1 : end + 1], dtype=torch.long)
        return x, y


def load_corpus(path: Path = TEXT_PATH) -> ShakespeareCorpus:
    return ShakespeareCorpus.from_file(path)


def build_dataloaders(corpus: ShakespeareCorpus, seq_len: int, batch_size: int, stride: int | None = None) -> tuple[DataLoader, DataLoader, DataLoader]:
    splits = corpus.split()
    train_ds = CharSequenceDataset(corpus.encode(splits["train"]), seq_len=seq_len, stride=stride)
    val_ds = CharSequenceDataset(corpus.encode(splits["validation"]), seq_len=seq_len, stride=stride)
    test_ds = CharSequenceDataset(corpus.encode(splits["test"]), seq_len=seq_len, stride=stride)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, val_loader, test_loader


## 3. Models

The three models share the same shape: embedding layer, recurrent layer, and a final linear layer that predicts the next character.


In [10]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 128, hidden_size: int = 256, dropout: float = 0.2) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids: torch.Tensor, hidden: torch.Tensor | None = None):
        x = self.dropout(self.embedding(input_ids))
        output, hidden = self.rnn(x, hidden)
        logits = self.fc(self.dropout(output))
        return logits, hidden

    def init_hidden(self, batch_size: int, device: torch.device) -> torch.Tensor:
        return torch.zeros(1, batch_size, self.hidden_size, device=device)


class LSTM1Layer(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 128, hidden_size: int = 256, dropout: float = 0.2) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids: torch.Tensor, hidden: tuple[torch.Tensor, torch.Tensor] | None = None):
        x = self.dropout(self.embedding(input_ids))
        output, hidden = self.lstm(x, hidden)
        logits = self.fc(self.dropout(output))
        return logits, hidden

    def init_hidden(self, batch_size: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
        h = torch.zeros(1, batch_size, self.hidden_size, device=device)
        c = torch.zeros(1, batch_size, self.hidden_size, device=device)
        return h, c


class LSTM2Layer(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 128, hidden_size: int = 256, dropout: float = 0.2) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=2, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids: torch.Tensor, hidden: tuple[torch.Tensor, torch.Tensor] | None = None):
        x = self.dropout(self.embedding(input_ids))
        output, hidden = self.lstm(x, hidden)
        logits = self.fc(self.dropout(output))
        return logits, hidden

    def init_hidden(self, batch_size: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
        h = torch.zeros(2, batch_size, self.hidden_size, device=device)
        c = torch.zeros(2, batch_size, self.hidden_size, device=device)
        return h, c


## 4. Training, evaluation, and text generation

The training loop uses cross-entropy loss, Adam, gradient clipping, validation loss, early stopping, and checkpoint saving.


In [11]:
def make_model(model_name: str, vocab_size: int, embedding_dim: int, hidden_size: int, dropout: float):
    if model_name == "rnn":
        return VanillaRNN(vocab_size, embedding_dim, hidden_size, dropout)
    if model_name == "lstm1":
        return LSTM1Layer(vocab_size, embedding_dim, hidden_size, dropout)
    if model_name == "lstm2":
        return LSTM2Layer(vocab_size, embedding_dim, hidden_size, dropout)
    raise ValueError(f"Unknown model name: {model_name}")


def perplexity_from_loss(loss: float) -> float:
    return float(math.exp(min(loss, 50.0)))


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_tokens = 0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        hidden = model.init_hidden(inputs.size(0), device)
        logits, _ = model(inputs, hidden)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        total_loss += float(loss.item()) * targets.numel()
        total_tokens += targets.numel()

    loss = total_loss / max(1, total_tokens)
    return {"loss": loss, "perplexity": perplexity_from_loss(loss)}


def ngram_overlap(generated: str, reference: str, n: int = 2) -> float:
    """Compute fraction of n-grams in generated text that appear in reference."""
    if len(generated) < n:
        return 0.0
    gen_ngrams = [generated[i:i+n] for i in range(len(generated) - n + 1)]
    ref_ngrams = set(reference[i:i+n] for i in range(len(reference) - n + 1))
    if not gen_ngrams:
        return 0.0
    overlap = sum(1 for ng in gen_ngrams if ng in ref_ngrams)
    return overlap / len(gen_ngrams)


def spelling_accuracy(text: str) -> float:
    """Compute fraction of words that are correctly spelled (simple check: contain mostly letters)."""
    import re
    words = re.findall(r"[a-zA-Z']+", text)
    if not words:
        return 0.0
    # Simple heuristic: a word is "correctly spelled" if it appears in training.
    # For now, we use a basic check: fraction of tokens that are alphabetic.
    correctly_spelled = sum(1 for w in words if len(w) > 2 and w.isalpha())
    return correctly_spelled / len(words) if words else 0.0


def top_p_sample(logits: torch.Tensor, temperature: float = 1.0, top_p: float = 1.0) -> int:
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    scaled = logits / temperature
    probabilities = torch.softmax(scaled, dim=-1)

    if top_p < 1.0:
        sorted_probabilities, sorted_indices = torch.sort(probabilities, descending=True)
        cumulative = torch.cumsum(sorted_probabilities, dim=-1)
        keep = cumulative <= top_p
        keep[0] = True
        filtered = sorted_probabilities * keep
        filtered = filtered / filtered.sum()
        choice = torch.multinomial(filtered, 1).item()
        return int(sorted_indices[choice].item())

    return int(torch.multinomial(probabilities, 1).item())


@torch.no_grad()
def generate_text(
    model: nn.Module,
    corpus: ShakespeareCorpus,
    prompt: str,
    length: int,
    temperature: float = 1.0,
    top_p: float = 1.0,
    device: torch.device | None = None,
) -> str:
    model.eval()
    device = device or next(model.parameters()).device
    prompt = prompt or corpus.text[:1]
    generated_ids = corpus.encode(prompt)

    input_ids = torch.tensor(generated_ids, dtype=torch.long, device=device).unsqueeze(0)
    hidden = model.init_hidden(1, device)
    logits, hidden = model(input_ids, hidden)

    for _ in range(length):
        next_logits = logits[:, -1, :].squeeze(0)
        next_id = top_p_sample(next_logits, temperature=temperature, top_p=top_p)
        generated_ids.append(next_id)
        step_input = torch.tensor([[next_id]], dtype=torch.long, device=device)
        logits, hidden = model(step_input, hidden)

    return corpus.decode(generated_ids)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = 10,
    learning_rate: float = 1e-3,
    grad_clip: float = 1.0,
    patience: int = 5,
    checkpoint_path: Path | None = None,
) -> list[dict[str, float]]:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history: list[dict[str, float]] = []
    best_val = float("inf")
    stalled = 0

    if checkpoint_path is not None:
        checkpoint_path.parent.mkdir(exist_ok=True)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_tokens = 0

        for inputs, targets in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
            inputs = inputs.to(device)
            targets = targets.to(device)
            hidden = model.init_hidden(inputs.size(0), device)

            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(inputs, hidden)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            train_loss += float(loss.item()) * targets.numel()
            train_tokens += targets.numel()

        train_loss = train_loss / max(1, train_tokens)
        val_metrics = evaluate_model(model, val_loader, device)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "train_perplexity": perplexity_from_loss(train_loss),
            "val_perplexity": val_metrics["perplexity"],
        }
        history.append(row)

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            stalled = 0
            if checkpoint_path is not None:
                torch.save({"model_state_dict": model.state_dict(), "history": history}, checkpoint_path)
        else:
            stalled += 1

        if patience and stalled >= patience:
            break

    return history


def character_bleu(reference: str, candidate: str) -> float:
    if not candidate:
        return 0.0
    smoothing = SmoothingFunction().method1
    return float(sentence_bleu([list(reference)], list(candidate), smoothing_function=smoothing))


## 5. Plots and experiments

We keep experiment tracking simple: train a few settings, save CSV results, and plot loss and perplexity curves.


In [12]:
def plot_history(history: list[dict[str, float]], output_dir: Path = OUTPUTS, prefix: str = "run") -> None:
    if not history:
        return

    epochs = [row["epoch"] for row in history]
    train_loss = [row["train_loss"] for row in history]
    val_loss = [row["val_loss"] for row in history]
    train_ppl = [row["train_perplexity"] for row in history]
    val_ppl = [row["val_perplexity"] for row in history]

    output_dir.mkdir(exist_ok=True)

    plt.figure(figsize=(10, 4))
    plt.plot(epochs, train_loss, label="train")
    plt.plot(epochs, val_loss, label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{prefix} loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / f"{prefix}_loss.png", dpi=150)
    plt.close()

    plt.figure(figsize=(10, 4))
    plt.plot(epochs, train_ppl, label="train")
    plt.plot(epochs, val_ppl, label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Perplexity")
    plt.title(f"{prefix} perplexity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / f"{prefix}_perplexity.png", dpi=150)
    plt.close()


def save_csv_rows(rows: list[dict[str, object]], path: Path) -> None:
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def run_experiment_grid(
    corpus: ShakespeareCorpus,
    seq_len: int,
    output_csv: Path,
    model_names: Iterable[str] = ("rnn", "lstm1", "lstm2"),
    hidden_sizes: Iterable[int] = (128, 256),
    learning_rates: Iterable[float] = (1e-3, 5e-4),
    batch_sizes: Iterable[int] = (32, 64),
    dropouts: Iterable[float] = (0.1, 0.2),
    epochs: int = 3,
) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []

    for model_name, hidden_size, lr, batch_size, dropout in product(model_names, hidden_sizes, learning_rates, batch_sizes, dropouts):
        train_loader, val_loader, test_loader = build_dataloaders(corpus, seq_len=seq_len, batch_size=batch_size, stride=seq_len)
        model = make_model(model_name, corpus.vocab_size, 128, hidden_size, dropout).to(device)
        checkpoint_path = OUTPUTS / f"{model_name}_h{hidden_size}_lr{lr}_bs{batch_size}_do{dropout}.pt"
        history = train_model(model, train_loader, val_loader, device, epochs=epochs, learning_rate=lr, checkpoint_path=checkpoint_path)
        test_metrics = evaluate_model(model, test_loader, device)
        rows.append({
            "model": model_name,
            "hidden_size": hidden_size,
            "learning_rate": lr,
            "batch_size": batch_size,
            "dropout": dropout,
            "best_val_loss": min(row["val_loss"] for row in history),
            "test_loss": test_metrics["loss"],
            "test_perplexity": test_metrics["perplexity"],
        })

    save_csv_rows(rows, output_csv)
    return rows


## 6. Example run



In [ ]:
# ============================================================================
# EXPERIMENT 1: Train and compare all three baselines
# ============================================================================
print("="*70)
print("EXPERIMENT 1: RNN vs LSTM-1L vs LSTM-2L (Baseline Comparison)")
print("="*70)

corpus = load_corpus(TEXT_PATH)
print(f"\nVocab size: {corpus.vocab_size}")
print(f"Total characters: {len(corpus.text)}")

# Use default settings
seq_len = 100
batch_size = 64
embedding_dim = 128
hidden_size = 256
dropout = 0.2
epochs = 3  # Quick run
learning_rate = 1e-3

train_loader, val_loader, test_loader = build_dataloaders(corpus, seq_len=seq_len, batch_size=batch_size, stride=seq_len)

models_to_train = [("rnn", VanillaRNN), ("lstm1", LSTM1Layer), ("lstm2", LSTM2Layer)]
baseline_results = {}
baseline_texts = {}

SEED_PROMPT = "To be, or not to be"
GEN_LENGTH = 200

for model_name, ModelClass in models_to_train:
    print(f"\n--- Training {model_name.upper()} ---")
    model = ModelClass(corpus.vocab_size, embedding_dim, hidden_size, dropout).to(device)
    checkpoint_path = OUTPUTS / f"{model_name}_baseline.pt"
    
    history = train_model(model, train_loader, val_loader, device, epochs=epochs, learning_rate=learning_rate, checkpoint_path=checkpoint_path)
    test_metrics = evaluate_model(model, test_loader, device)
    
    # Generate text
    gen_text = generate_text(model, corpus, SEED_PROMPT, GEN_LENGTH, temperature=0.8, top_p=0.95, device=device)
    baseline_texts[model_name] = gen_text
    
    baseline_results[model_name] = {
        "test_loss": test_metrics["loss"],
        "test_perplexity": test_metrics["perplexity"],
        "params": sum(p.numel() for p in model.parameters()),
        "bigram_overlap": ngram_overlap(gen_text, corpus.text, 2),
        "trigram_overlap": ngram_overlap(gen_text, corpus.text, 3),
        "bleu": character_bleu(corpus.text[:1000], gen_text),
        "spelling": spelling_accuracy(gen_text),
    }

# Print comparison table
print("\n" + "="*70)
print("QUANTITATIVE COMPARISON")
print("="*70)
print(f"{'Model':<8} | {'Perplexity':>10} | {'BLEU':>8} | {'Bigram':>8} | {'Trigram':>8} | {'Spelling':>8} | {'Params':>10}")
print("-"*90)
for mname, res in baseline_results.items():
    print(f"{mname:<8} | {res['test_perplexity']:>10.2f} | {res['bleu']:>8.4f} | {res['bigram_overlap']:>8.3f} | {res['trigram_overlap']:>8.3f} | {res['spelling']:>8.3f} | {res['params']:>10,}")

# Qualitative samples
print("\n" + "="*70)
print("QUALITATIVE TEXT SAMPLES (temperature=0.8, top_p=0.95)")
print("="*70)
for mname, text in baseline_texts.items():
    print(f"\n--- {mname.upper()} ---")
    print(text[:300] + "...")


# ============================================================================
# EXPERIMENT 2: Hidden size investigation
# ============================================================================
print("\n\n" + "="*70)
print("EXPERIMENT 2: Effect of Hidden Size on Performance")
print("="*70)

hidden_sizes = [64, 128, 256, 512]
hidden_results = []

for h_size in hidden_sizes:
    print(f"\nTesting hidden_size={h_size}...")
    train_loader, val_loader, test_loader = build_dataloaders(corpus, seq_len=seq_len, batch_size=batch_size)
    model = LSTM1Layer(corpus.vocab_size, embedding_dim, h_size, dropout).to(device)
    
    history = train_model(model, train_loader, val_loader, device, epochs=2, learning_rate=learning_rate, patience=3)
    test_metrics = evaluate_model(model, test_loader, device)
    gen_text = generate_text(model, corpus, SEED_PROMPT, GEN_LENGTH, temperature=0.8, device=device)
    
    hidden_results.append({
        "hidden_size": h_size,
        "test_perplexity": test_metrics["perplexity"],
        "params": sum(p.numel() for p in model.parameters()),
        "bleu": character_bleu(corpus.text[:1000], gen_text),
    })

print("\n" + "="*70)
print("HIDDEN SIZE RESULTS (1-Layer LSTM)")
print("="*70)
print(f"{'Hidden Size':>12} | {'Perplexity':>10} | {'BLEU':>8} | {'Parameters':>12}")
print("-"*50)
for r in hidden_results:
    print(f"{r['hidden_size']:>12} | {r['test_perplexity']:>10.2f} | {r['bleu']:>8.4f} | {r['params']:>12,}")


# ============================================================================
# EXPERIMENT 3: Hyperparameter Grid Search
# ============================================================================
print("\n\n" + "="*70)
print("EXPERIMENT 3: Hyperparameter Grid Search")
print("="*70)
print("(batch_size × learning_rate)")

batch_sizes_grid = [32, 64]
learning_rates_grid = [5e-4, 1e-3]
grid_results = []

for bs in batch_sizes_grid:
    for lr in learning_rates_grid:
        print(f"\nTesting batch_size={bs}, learning_rate={lr}...")
        train_ldr, val_ldr, test_ldr = build_dataloaders(corpus, seq_len=seq_len, batch_size=bs)
        model = LSTM1Layer(corpus.vocab_size, embedding_dim, hidden_size, dropout).to(device)
        
        history = train_model(model, train_ldr, val_ldr, device, epochs=2, learning_rate=lr, patience=2)
        test_metrics = evaluate_model(model, test_ldr, device)
        
        grid_results.append({
            "batch_size": bs,
            "learning_rate": lr,
            "test_perplexity": test_metrics["perplexity"],
            "best_val_loss": min(h["val_loss"] for h in history),
        })

print("\n" + "="*70)
print("HYPERPARAMETER GRID RESULTS")
print("="*70)
print(f"{'Batch Size':>12} | {'Learning Rate':>15} | {'Test Perplexity':>15} | {'Best Val Loss':>14}")
print("-"*60)
for r in sorted(grid_results, key=lambda x: x['test_perplexity']):
    print(f"{r['batch_size']:>12} | {r['learning_rate']:>15.5f} | {r['test_perplexity']:>15.2f} | {r['best_val_loss']:>14.4f}")

best_hp = min(grid_results, key=lambda x: x['test_perplexity'])
print(f"\nBest hyperparameters: batch_size={best_hp['batch_size']}, lr={best_hp['learning_rate']}")
print(f"Best test perplexity: {best_hp['test_perplexity']:.2f}")


# ============================================================================
# EXPERIMENT 4: Temperature and Nucleus Sampling
# ============================================================================
print("\n\n" + "="*70)
print("EXPERIMENT 4: Temperature and Nucleus Sampling Investigation")
print("="*70)

# Use the best model from baseline
best_model = LSTM1Layer(corpus.vocab_size, embedding_dim, hidden_size, dropout).to(device)
history = train_model(best_model, train_loader, val_loader, device, epochs=2, learning_rate=learning_rate)

print("\n--- Temperature (T=0.5: deterministic, T=1.5: random) ---")
for temp in [0.5, 1.0, 1.5]:
    gen = generate_text(best_model, corpus, SEED_PROMPT, 150, temperature=temp, top_p=1.0, device=device)
    print(f"\nT={temp}: {gen[:100]}...")

print("\n--- Nucleus Sampling (top_p) with fixed T=1.0 ---")
for top_p in [0.7, 0.9, 0.95]:
    gen = generate_text(best_model, corpus, SEED_PROMPT, 150, temperature=1.0, top_p=top_p, device=device)
    print(f"\ntop_p={top_p}: {gen[:100]}...")

print("\n" + "="*70)
print("ALL EXPERIMENTS COMPLETE")
print("="*70)


EXPERIMENT 1: RNN vs LSTM-1L vs LSTM-2L (Baseline Comparison)

Vocab size: 65
Total characters: 1115393

--- Training RNN ---



--- Training LSTM1 ---



--- Training LSTM2 ---



QUANTITATIVE COMPARISON
Model    | Perplexity |     BLEU |   Bigram |  Trigram | Spelling |     Params
------------------------------------------------------------------------------------------
rnn      |       7.04 |   0.0174 |    0.995 |    0.991 |    0.640 |    123,841
lstm1    |       6.74 |   0.0139 |    1.000 |    0.982 |    0.705 |    420,289
lstm2    |       6.59 |   0.0171 |    1.000 |    0.986 |    0.729 |    946,625

QUALITATIVE TEXT SAMPLES (temperature=0.8, top_p=0.95)

--- RNN ---
To be, or not to be be crefur we came with him were is to but the kenter hem a ang there the arest on he mane the bones all the would the ward in the gilling the lord?

DUCHAS:
Oy word's to hith my lood, in all the deet...

--- LSTM1 ---
To be, or not to be light,
Fre prientle sack mow you my her werse they last us.

CAMELLEN:
My beting death my falles.

FLORCHESS:
Here that shall by the rid him to may of him,
Which when lown we for mesting connot made ...

--- LSTM2 ---
To be, or not to bend,



Testing hidden_size=128...



Testing hidden_size=256...


epoch 2:  38%|███▊      | 53/139 [00:16<00:25,  3.39it/s] 